# Read from Bronze Table

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
display(df)

# Data Transformations

In [0]:
rename_mapping = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date",
}

# trim name, surname etc.
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, F.trim(F.col(field.name)))

# normalization of marital status, gndr
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
        .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
        .when(F.upper(F.col("cst_gndr")) == "M", "Male")
        .otherwise("n/a")
    )
)

# column names are not friendly
for old_name, new_name in rename_mapping.items():
    df = df.withColumnRenamed(old_name, new_name)

display(df)

# Write Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")